# Factory Dataset Exploration

Exploratory analysis of the synthetic production dataset used by the
reinforcement learning environments: 3,000 days of demand and resource
availability for products A, B and C.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd().parent
DATA_PATH = BASE_DIR / "data" / "raw" / "production_data.csv"

df = pd.read_csv(DATA_PATH)
df.shape

In [ ]:
df.head()

## 1. Demand statistics

In [ ]:
df[["demand_A", "demand_B", "demand_C"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, product in zip(axes, ["A", "B", "C"]):
    ax.hist(df[f"demand_{product}"], bins=30, edgecolor="black")
    ax.set_title(f"Product {product} demand")
    ax.set_xlabel("Units/day")
    ax.set_ylabel("Days")

plt.tight_layout()
plt.show()

## 2. Demand over time

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

window = df.head(180)

ax.plot(window["day"], window["demand_A"], label="A")
ax.plot(window["day"], window["demand_B"], label="B")
ax.plot(window["day"], window["demand_C"], label="C")

ax.set_xlabel("Day")
ax.set_ylabel("Units")
ax.set_title("Daily demand (first 180 days)")
ax.legend()
plt.show()

## 3. Weekly demand pattern

The generator applies a weekly cycle of +/-10% plus a slower
90-day seasonal component.

In [ ]:
weekly = df.groupby((df["day"] - 1) % 7)[
    ["demand_A", "demand_B", "demand_C"]
].mean()

weekly.plot(marker="o", figsize=(8, 4))
plt.xlabel("Day of week")
plt.ylabel("Mean demand")
plt.title("Weekly demand pattern")
plt.show()

## 4. Resource availability

In [ ]:
df[[
    "raw_material_capacity",
    "machine_hours_capacity",
    "production_budget"
]].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

window = df.head(180)

ax.plot(window["day"], window["raw_material_capacity"], label="Raw material")
ax.plot(window["day"], window["machine_hours_capacity"] * 1.8, label="Machine hours (x1.8)")

ax.set_xlabel("Day")
ax.set_ylabel("Units")
ax.set_title("Daily resource capacity (first 180 days)")
ax.legend()
plt.show()

## 5. Product economics

Margin per unit and per unit of each constrained resource.

In [ ]:
economics = pd.DataFrame({
    "price": [500, 400, 300],
    "cost": [250, 180, 120],
    "material": [3, 2, 1],
    "hours": [2.0, 1.5, 1.0]
}, index=["A", "B", "C"])

economics["margin"] = economics["price"] - economics["cost"]
economics["margin_per_material"] = economics["margin"] / economics["material"]
economics["margin_per_hour"] = economics["margin"] / economics["hours"]

economics

## 6. Constraint check at average demand

Meeting average demand exactly would require more machine hours than
the factory typically has — machine hours are the binding constraint,
which is why production plans that over-allocate hours to one product
lose profit.

In [ ]:
avg_demand = df[["demand_A", "demand_B", "demand_C"]].mean().to_numpy()

hours_needed = float(np.dot(avg_demand, [2.0, 1.5, 1.0]))
material_needed = float(np.dot(avg_demand, [3, 2, 1]))
budget_needed = float(np.dot(avg_demand, [250, 180, 120]))

print(f"Machine hours needed: {hours_needed:.1f}  (available ~{df['machine_hours_capacity'].mean():.0f})")
print(f"Raw material needed: {material_needed:.1f}  (available ~{df['raw_material_capacity'].mean():.0f})")
print(f"Budget needed: {budget_needed:,.0f}  (available ~{df['production_budget'].mean():,.0f})")